In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.linear_model import LogisticRegression

In [18]:
df = pd.read_csv("data/islp_default.csv")
df.info()
df.describe()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   default  10000 non-null  object 
 1   student  10000 non-null  object 
 2   balance  10000 non-null  float64
 3   income   10000 non-null  float64
dtypes: float64(2), object(2)
memory usage: 312.6+ KB


,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138947
3,No,No,529.250605,35704.493935
4,No,No,785.655883,38463.495879


In [11]:
df["student"].value_counts()

student
No     7056
Yes    2944
Name: count, dtype: int64

In [12]:
df["default"].value_counts() # data is quite skewed - 9667 no default vs 333 yes default

default
No     9667
Yes     333
Name: count, dtype: int64

In [39]:
# stupid null model
baseline = 9667 / (9667 + 333)
print(baseline)

0.9667


In [4]:
# Some preprocessing needed

In [21]:
df = pd.read_csv("data/islp_default.csv")
df = pd.get_dummies(df, columns = ["student", "default"], dtype = int)
df = df[["balance", "income", "student_Yes", "default_Yes"]]
df.info()
df.describe()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   balance      10000 non-null  float64
 1   income       10000 non-null  float64
 2   student_Yes  10000 non-null  int64  
 3   default_Yes  10000 non-null  int64  
dtypes: float64(2), int64(2)
memory usage: 312.6 KB


,balance,income,student_Yes,default_Yes
0,729.526495,44361.625074,0,0
1,817.180407,12106.134700,1,0
2,1073.549164,31767.138947,0,0
3,529.250605,35704.493935,0,0
4,785.655883,38463.495879,0,0


In [28]:
# Basic sklearn LogisticRegression model to check answers against

predictors = ["balance", "income", "student_Yes"]
targets = "default_Yes"

X_train = np.array(df[predictors])
Y_train = np.array(df[targets])

model = LogisticRegression()
model.fit(X_train, Y_train)

beta0 = model.intercept_
betap = model.coef_
score = model.score(X_train, Y_train)

print("beta0, betap = %s, %s" % (beta0, betap))
print("accuracy = %s" % score)

beta0, betap = [-10.9018116], [[ 5.73060606e-03  3.96189927e-06 -6.12564509e-01]]
accuracy = 0.9732


In [42]:
# homemade logistic regression function

def my_logistic_regression(X_train, Y_train):
    
    X_train = np.insert(X_train, 0, 1, axis = 1)
    
    N = len(X_train)
    
#     beta = np.array([0.1, 0.2, 0.3, 0.4]).reshape((-1, 1))
    beta = np.array([-10.9018116, 0.00573060606,  0.00000396189927, -0.612564509]).reshape((-1, 1))
    
    probabilities = []
    for i in range(N):
        b = (X_train[i] @ beta).item()
        if (b > 0):
            probabilities.append(1/(1+np.exp(-b)))
        else:
            probabilities.append(np.exp(b)/(1+np.exp(b)))
    probabilities = np.array(probabilities)

    Yhat = (probabilities < 0.5)
    accuracy = np.mean(np.abs(Y_train - Yhat))
    
    print("accuracy = %s" % accuracy)

In [43]:
predictors = ["balance", "income", "student_Yes"]
targets = "default_Yes"

X_train = np.array(df[predictors])
Y_train = np.array(df[targets])

my_logistic_regression(X_train, Y_train)

accuracy = 0.9732
